# 🎙️ Voice Over Studio (Bahasa Indonesia) — Colab All-in-One

Aplikasi voice over cloning suara **langsung di dalam Colab**. Tidak perlu
menyalin URL ke mana-mana, tidak perlu file `index.html`.

**Cara pakai:**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. Tunggu sampai muncul **link `https://xxxx.gradio.live`** di bawah sel ke-2.
   **Klik link itu** untuk membuka aplikasinya di tab baru (mic lebih lancar di sana).
4. Di aplikasi: rekam/unggah sampel suara, tulis naskah, klik **Buat Voice Over**.

> Gunakan hanya suara Anda sendiri / narator yang sudah mengizinkan.

In [ ]:
# 1) Pasang program (tunggu 2-4 menit)
import subprocess, sys
print('Memasang F5-TTS + Gradio, mohon tunggu...')
subprocess.run([sys.executable,'-m','pip','install','-q','f5-tts','gradio'])
subprocess.run(['bash','-c','apt-get -qq install -y ffmpeg >/dev/null 2>&1'])
print('OK, pemasangan selesai.')

In [ ]:
# 2) Jalankan aplikasi (muat model + tampilkan antarmuka)
import torch, gradio as gr
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

print("Mengunduh & memuat model Indonesia (1-2 menit, sekali saja)...")
ckpt = hf_hub_download('Eempostor/F5-TTS-INDO-FINETUNE-V2', 'f5_tts_indo_v2.pt')
vocab = hf_hub_download('Eempostor/F5-TTS-INDO-FINETUNE-V2', 'vocab.txt')
MODEL = F5TTS(model='F5TTS_v1_Base', ckpt_file=ckpt, vocab_file=vocab)
print("Model siap.")

def buat_suara(ref_audio, ref_text, naskah, kecepatan):
    if ref_audio is None:
        raise gr.Error("Rekam atau unggah sampel suara dulu.")
    if not naskah or not naskah.strip():
        raise gr.Error("Tulis naskah voice over dulu.")
    wav, sr, _ = MODEL.infer(
        ref_file=ref_audio,
        ref_text=(ref_text or "").strip(),
        gen_text=naskah.strip(),
        speed=float(kecepatan),
        remove_silence=True,
    )
    return (sr, wav)

with gr.Blocks(title="Voice Over Studio - Portal BMP") as demo:
    gr.Markdown("# 🎙️ Voice Over Studio (Bahasa Indonesia)\n"
                "Cloning suara dengan F5-TTS. **Gunakan hanya suara Anda sendiri "
                "atau narator yang sudah mengizinkan.**")
    with gr.Row():
        with gr.Column():
            ref = gr.Audio(label="1. Sampel suara (rekam / unggah, 10-20 detik)",
                           sources=["microphone", "upload"], type="filepath")
            reft = gr.Textbox(label="Transkrip sampel (opsional, bikin lebih akurat)",
                              placeholder="Kalimat persis yang diucapkan di sampel...")
            naskah = gr.Textbox(label="2. Naskah voice over", lines=6,
                                placeholder="Tempel naskah berita di sini...")
            spd = gr.Slider(0.6, 1.4, value=1.0, step=0.05, label="Kecepatan")
            btn = gr.Button("Buat Voice Over", variant="primary")
        with gr.Column():
            out = gr.Audio(label="Hasil voice over (bisa diputar & diunduh)",
                           type="numpy")
    btn.click(buat_suara, [ref, reft, naskah, spd], out)

demo.launch(share=True)
